# 00 Data Sanity

Purpose:
- Load or build pair feature panels
- Check shape, date coverage, missingness, and return sanity
- Visualize prices and return distributions


In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Ensure all project-relative IO uses repository root
os.chdir(ROOT)

from src.utils.io import load_config
from src.experiments import build_processed_data, pair_list


In [ ]:
CONFIG_PATH = ROOT / "configs" / "default.yaml"
TRY_BUILD_IF_MISSING = True

cfg = load_config(CONFIG_PATH)
cfg["evaluation"]["plot"] = False
cfg["evaluation"]["save_dir"] = str(ROOT / "reports")

pairs = pair_list(cfg)
pair_names = [p["name"] for p in pairs]
print("Pairs:", pair_names)


In [ ]:
def load_features_local_or_build(cfg: dict, try_build: bool = True) -> dict[str, pd.DataFrame]:
    processed_dir = ROOT / "data" / "processed"
    paths = {
        p["name"]: processed_dir / f"{p['name']}_features.csv"
        for p in pair_list(cfg)
    }

    if all(path.exists() for path in paths.values()):
        out = {}
        for name, path in paths.items():
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            df.index = pd.to_datetime(df.index, utc=True)
            out[name] = df
        return out

    if not try_build:
        missing = [str(p) for p in paths.values() if not p.exists()]
        raise FileNotFoundError(f"Missing processed files: {missing}")

    print("Processed files missing. Building from loaders (may trigger downloads)...")
    return build_processed_data(cfg)


In [ ]:
features_by_pair = load_features_local_or_build(cfg, try_build=TRY_BUILD_IF_MISSING)
for name, df in features_by_pair.items():
    print(name, df.shape, df.index.min(), "->", df.index.max())


In [ ]:
rows = []
for pair in pairs:
    name = pair["name"]
    df = features_by_pair[name]
    rows.append({
        "pair": name,
        "rows": len(df),
        "start": df.index.min(),
        "end": df.index.max(),
        "missing_total": int(df.isna().sum().sum()),
        "spot_price_min": float(df["spot_adj_close"].min()),
        "lev_price_min": float(df["lev_adj_close"].min()),
        "spot_ret_std": float(df["spot_ret"].std()),
        "lev_ret_std": float(df["lev_ret"].std()),
    })

summary = pd.DataFrame(rows).set_index("pair").sort_index()
summary


In [ ]:
for pair in pairs:
    name = pair["name"]
    df = features_by_pair[name]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(df.index, df["spot_adj_close"], label="Spot", lw=1.5)
    axes[0].plot(df.index, df["lev_adj_close"], label="Leveraged", lw=1.2)
    axes[0].set_title(f"{name}: adjusted closes")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    axes[1].hist(df["spot_ret"], bins=50, alpha=0.6, label="Spot")
    axes[1].hist(df["lev_ret"], bins=50, alpha=0.6, label="Leveraged")
    axes[1].set_title(f"{name}: daily return distribution")
    axes[1].legend()
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


In [ ]:
required_cols = ["spot_adj_close", "lev_adj_close", "spot_ret", "lev_ret", "drift", "vol", "beta"]

for name, df in features_by_pair.items():
    assert not df.empty, f"{name}: empty dataframe"
    assert set(required_cols).issubset(df.columns), f"{name}: missing required columns"
    assert (df["spot_adj_close"] > 0).all(), f"{name}: non-positive spot prices"
    assert (df["lev_adj_close"] > 0).all(), f"{name}: non-positive leveraged prices"
    assert df[required_cols].isna().sum().sum() == 0, f"{name}: NaNs remain in required cols"

print("Data sanity checks passed.")
